# Analyse Exploratoire — Rossmann Store Sales

Ce notebook explore le dataset Rossmann enrichi pour comprendre les patterns de ventes
avant la phase de modélisation. On examine :

1. Distribution générale des ventes
2. Patterns temporels (saisonnalité, jour de la semaine)
3. Effet des promotions
4. Impact de la météo
5. Différences entre types de magasins
6. Outliers et anomalies
7. Corrélations entre features

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL

from src.config import PROCESSED_DIR, RAW_DIR

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 100

%matplotlib inline

## 1. Chargement des données

In [ ]:
# Charger les données enrichies (sortie de la Phase 1)
enriched_path = PROCESSED_DIR / "rossmann_enriched.parquet"

if enriched_path.exists():
    df = pd.read_parquet(enriched_path)
    print(f"Données enrichies : {len(df):,} lignes, {len(df.columns)} colonnes")
else:
    # Fallback sur les données nettoyées
    df = pd.read_parquet(RAW_DIR / "rossmann_clean.parquet")
    print(f"Données nettoyées (non enrichies) : {len(df):,} lignes")

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(f"Période : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Magasins : {df['store_id'].nunique()}")
print(f"Valeurs manquantes :")
missing = df.isnull().sum()
print(missing[missing > 0])

## 2. Distribution des ventes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribution brute
axes[0].hist(df["sales"], bins=80, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(df["sales"].median(), color="red", linestyle="--", label=f"Médiane = {df['sales'].median():,.0f}")
axes[0].set_title("Distribution des ventes")
axes[0].set_xlabel("Ventes")
axes[0].legend()

# Distribution log
axes[1].hist(np.log1p(df["sales"]), bins=80, color="teal", edgecolor="white", alpha=0.85)
axes[1].set_title("Distribution log(1 + ventes)")
axes[1].set_xlabel("log(1 + sales)")

# Boxplot par type de magasin
if "store_type" in df.columns:
    df.boxplot(column="sales", by="store_type", ax=axes[2])
    axes[2].set_title("Ventes par type de magasin")
    axes[2].set_xlabel("Type")
    axes[2].set_ylabel("Ventes")
    plt.suptitle("")

plt.tight_layout()
plt.show()

In [ ]:
# Ventes moyennes par magasin
store_means = df.groupby("store_id")["sales"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(16, 4))
ax.bar(range(len(store_means)), store_means.values, color="steelblue", width=1.0)
ax.set_title(f"Ventes moyennes par magasin (n={len(store_means)})")
ax.set_xlabel("Magasin (trié)")
ax.set_ylabel("Ventes moyennes")
plt.tight_layout()
plt.show()

print(f"Top 5 magasins : {store_means.head().to_dict()}")
print(f"Bottom 5 magasins : {store_means.tail().to_dict()}")

## 3. Patterns temporels

In [ ]:
# Ventes moyennes par jour de la semaine
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

dow_sales = df.groupby("day_of_week")["sales"].mean()
dow_labels = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]
axes[0].bar(dow_sales.index, dow_sales.values, color="steelblue")
axes[0].set_xticks(range(1, 8))
axes[0].set_xticklabels(dow_labels)
axes[0].set_title("Ventes moyennes par jour de la semaine")
axes[0].set_ylabel("Ventes")

# Ventes moyennes par mois
if "month" in df.columns:
    month_sales = df.groupby("month")["sales"].mean()
    axes[1].bar(month_sales.index, month_sales.values, color="teal")
    axes[1].set_xticks(range(1, 13))
    axes[1].set_title("Ventes moyennes par mois")
    axes[1].set_ylabel("Ventes")

plt.tight_layout()
plt.show()

In [ ]:
# Série temporelle agrégée (ventes totales par jour)
daily_total = df.groupby("date")["sales"].sum()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily_total.index, daily_total.values, linewidth=0.6, color="steelblue")
ax.set_title("Ventes totales journalières (tous magasins)")
ax.set_xlabel("Date")
ax.set_ylabel("Ventes totales")
plt.tight_layout()
plt.show()

In [ ]:
# Décomposition STL sur un magasin exemple
sample_store = df[df["store_id"] == 1].set_index("date")["sales"].sort_index()
sample_store = sample_store.asfreq("D").interpolate()

stl = STL(sample_store, period=7, seasonal=13, robust=True)
result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True)
axes[0].plot(result.observed, linewidth=0.6)
axes[0].set_ylabel("Observé")
axes[0].set_title("Décomposition STL — Magasin 1 (période=7 jours)")

axes[1].plot(result.trend, color="orange", linewidth=1)
axes[1].set_ylabel("Tendance")

axes[2].plot(result.seasonal, color="green", linewidth=0.6)
axes[2].set_ylabel("Saisonnalité")

axes[3].plot(result.resid, color="red", linewidth=0.4, alpha=0.7)
axes[3].set_ylabel("Résidus")

plt.tight_layout()
plt.show()

## 4. Effet des promotions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Promo vs pas promo
promo_sales = df.groupby("promo")["sales"].mean()
axes[0].bar(["Sans promo", "Avec promo"], promo_sales.values, color=["#94a3b8", "#f59e0b"])
axes[0].set_title("Ventes moyennes : promo vs sans promo")
axes[0].set_ylabel("Ventes")
lift = (promo_sales[1] - promo_sales[0]) / promo_sales[0] * 100
axes[0].annotate(f"+{lift:.1f}%", xy=(1, promo_sales[1]), fontsize=14,
                 ha="center", va="bottom", fontweight="bold", color="#d97706")

# Distribution avec/sans promo
df[df["promo"] == 0]["sales"].hist(bins=60, alpha=0.6, label="Sans promo", ax=axes[1], color="#94a3b8")
df[df["promo"] == 1]["sales"].hist(bins=60, alpha=0.6, label="Avec promo", ax=axes[1], color="#f59e0b")
axes[1].set_title("Distribution des ventes par statut promo")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Effet promo par jour de la semaine
promo_dow = df.groupby(["day_of_week", "promo"])["sales"].mean().unstack()
promo_dow.columns = ["Sans promo", "Avec promo"]

promo_dow.plot(kind="bar", figsize=(12, 5), color=["#94a3b8", "#f59e0b"])
plt.title("Ventes moyennes par jour × promo")
plt.xlabel("Jour de la semaine")
plt.ylabel("Ventes")
plt.xticks(range(7), ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"], rotation=0)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Effet promo par type de magasin
if "store_type" in df.columns:
    promo_type = df.groupby(["store_type", "promo"])["sales"].mean().unstack()
    promo_type.columns = ["Sans promo", "Avec promo"]
    promo_type["Lift %"] = (promo_type["Avec promo"] - promo_type["Sans promo"]) / promo_type["Sans promo"] * 100
    print("Effet promo par type de magasin :")
    print(promo_type.round(1))

## 5. Impact de la météo

In [ ]:
if "temperature_mean" in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Scatter : température vs ventes
    sample = df.sample(min(10_000, len(df)), random_state=42)
    axes[0].scatter(sample["temperature_mean"], sample["sales"], alpha=0.1, s=5, color="steelblue")
    axes[0].set_title("Température vs ventes")
    axes[0].set_xlabel("Température moyenne (°C)")
    axes[0].set_ylabel("Ventes")

    # Ventes par tranches de température
    df["temp_bin"] = pd.cut(df["temperature_mean"], bins=[-20, 0, 10, 20, 30, 40], labels=["<0", "0-10", "10-20", "20-30", "30+"])
    temp_sales = df.groupby("temp_bin", observed=True)["sales"].mean()
    temp_sales.plot(kind="bar", ax=axes[1], color="teal")
    axes[1].set_title("Ventes moyennes par tranche de température")
    axes[1].set_xlabel("Température (°C)")
    axes[1].tick_params(axis="x", rotation=0)

    # Pluie vs pas pluie
    if "is_rainy" in df.columns:
        rain_sales = df.groupby("is_rainy")["sales"].mean()
        axes[2].bar(["Sec", "Pluie (>1mm)"], rain_sales.values, color=["#f59e0b", "#3b82f6"])
        axes[2].set_title("Ventes : jours secs vs pluvieux")
        axes[2].set_ylabel("Ventes")

    df.drop(columns="temp_bin", inplace=True)
    plt.tight_layout()
    plt.show()
else:
    print("Pas de données météo — lancer d'abord src.data.enrichment")

## 6. Jours fériés

In [ ]:
if "is_holiday" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Férié vs normal
    hol_sales = df.groupby("is_holiday")["sales"].mean()
    axes[0].bar(["Normal", "Jour férié"], hol_sales.values, color=["#94a3b8", "#ef4444"])
    axes[0].set_title("Ventes moyennes : jour normal vs férié")
    axes[0].set_ylabel("Ventes")

    # Ventes vs proximité du jour férié
    if "days_to_holiday" in df.columns:
        prox = df[df["days_to_holiday"] <= 14].groupby("days_to_holiday")["sales"].mean()
        axes[1].plot(prox.index, prox.values, marker="o", color="#ef4444")
        axes[1].set_title("Ventes vs jours avant/après le férié le plus proche")
        axes[1].set_xlabel("Jours jusqu'au férié")
        axes[1].set_ylabel("Ventes moyennes")
        axes[1].axvline(0, color="grey", linestyle="--", alpha=0.5)

    plt.tight_layout()
    plt.show()
else:
    print("Pas de features jours fériés")

## 7. Différences entre types de magasins

In [ ]:
if "store_type" in df.columns and "assortment" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Par store_type
    type_stats = df.groupby("store_type").agg(
        mean_sales=("sales", "mean"),
        mean_customers=("customers", "mean"),
        n_stores=("store_id", "nunique"),
    ).round(0)

    type_stats["mean_sales"].plot(kind="bar", ax=axes[0], color="steelblue")
    axes[0].set_title("Ventes moyennes par type")
    axes[0].tick_params(axis="x", rotation=0)
    for i, (idx, row) in enumerate(type_stats.iterrows()):
        axes[0].annotate(f"n={int(row['n_stores'])}", xy=(i, row["mean_sales"]),
                         ha="center", va="bottom", fontsize=9, color="grey")

    # Par assortiment
    assort_stats = df.groupby("assortment")["sales"].mean()
    assort_stats.plot(kind="bar", ax=axes[1], color="teal")
    axes[1].set_title("Ventes moyennes par assortiment")
    axes[1].tick_params(axis="x", rotation=0)

    plt.tight_layout()
    plt.show()

    print("\nStatistiques par type de magasin :")
    print(type_stats)

In [ ]:
# Relation clients / ventes
if "customers" in df.columns:
    sample = df.sample(min(15_000, len(df)), random_state=42)

    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(sample["customers"], sample["sales"], alpha=0.1, s=5, c=sample["promo"],
                         cmap="coolwarm", vmin=0, vmax=1)
    ax.set_title("Clients vs ventes (couleur = promo)")
    ax.set_xlabel("Clients")
    ax.set_ylabel("Ventes")
    plt.colorbar(scatter, label="Promo")
    plt.tight_layout()
    plt.show()

    corr = df[["sales", "customers"]].corr().iloc[0, 1]
    print(f"Corrélation ventes/clients : {corr:.4f}")

## 8. Détection d'outliers

In [ ]:
# Outliers par magasin (ventes > 3σ de la moyenne du magasin)
store_stats = df.groupby("store_id")["sales"].agg(["mean", "std"]).reset_index()
store_stats.columns = ["store_id", "store_mean", "store_std"]
df_check = df.merge(store_stats, on="store_id")
df_check["z_score"] = (df_check["sales"] - df_check["store_mean"]) / df_check["store_std"]

outliers = df_check[df_check["z_score"].abs() > 3]
print(f"Outliers (|z| > 3) : {len(outliers):,} lignes ({len(outliers)/len(df)*100:.2f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_check["z_score"], bins=100, color="steelblue", edgecolor="white")
axes[0].axvline(-3, color="red", linestyle="--", alpha=0.7)
axes[0].axvline(3, color="red", linestyle="--", alpha=0.7)
axes[0].set_title("Distribution des z-scores (par magasin)")
axes[0].set_xlabel("Z-score")

outliers_per_store = outliers.groupby("store_id").size().sort_values(ascending=False).head(15)
outliers_per_store.plot(kind="bar", ax=axes[1], color="#ef4444")
axes[1].set_title("Top 15 magasins avec le plus d'outliers")
axes[1].set_xlabel("Store ID")
axes[1].set_ylabel("Nombre d'outliers")

plt.tight_layout()
plt.show()

## 9. Matrice de corrélation

In [ ]:
# Sélectionner les features numériques
num_cols = [
    "sales", "customers", "promo", "day_of_week",
    "is_weekend", "is_holiday", "days_to_holiday",
    "month", "competition_distance",
    "temperature_mean", "precipitation", "is_rainy",
]
available = [c for c in num_cols if c in df.columns]
corr_matrix = df[available].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title("Matrice de corrélation")
plt.tight_layout()
plt.show()

In [ ]:
# Corrélations avec la cible (sales)
if "sales" in corr_matrix.columns:
    target_corr = corr_matrix["sales"].drop("sales").sort_values(key=abs, ascending=False)
    print("Corrélation avec les ventes (par ordre d'importance) :")
    print(target_corr.round(4).to_string())

## 10. Vérification de la compétition

In [ ]:
if "competition_distance" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Distribution des distances
    axes[0].hist(df["competition_distance"].dropna(), bins=60, color="steelblue", edgecolor="white")
    axes[0].set_title("Distance au concurrent le plus proche")
    axes[0].set_xlabel("Distance (m)")

    # Distance vs ventes moyennes par magasin
    store_data = df.groupby("store_id").agg(
        mean_sales=("sales", "mean"),
        comp_dist=("competition_distance", "first"),
    ).dropna()

    axes[1].scatter(store_data["comp_dist"], store_data["mean_sales"], alpha=0.5, s=20, color="teal")
    axes[1].set_title("Distance concurrent vs ventes moyennes")
    axes[1].set_xlabel("Distance concurrent (m)")
    axes[1].set_ylabel("Ventes moyennes")

    plt.tight_layout()
    plt.show()

## 11. Feature Engineering — aperçu

Exécuter `python -m src.features.engineering` pour générer le dataset avec toutes les features.

Voici un aperçu des features qui seront ajoutées :

In [ ]:
from src.features.engineering import build_features

# Générer les features sur un sous-ensemble pour vérification
sample_stores = df[df["store_id"].isin([1, 2, 3])].copy()
features_sample = build_features(sample_stores, drop_na=True)

print(f"Colonnes après feature engineering ({len(features_sample.columns)}) :")
for col in sorted(features_sample.columns):
    print(f"  {col}")

print(f"\n{len(features_sample)} lignes (après suppression des NaN de warm-up)")

In [ ]:
# Corrélation des nouvelles features avec sales
new_cols = [c for c in features_sample.columns if "lag" in c or "rolling" in c or "ewm" in c or "trend" in c]
if new_cols:
    new_corr = features_sample[new_cols + ["sales"]].corr()["sales"].drop("sales").sort_values(key=abs, ascending=False)
    print("Corrélation des features lag/rolling avec les ventes :")
    print(new_corr.round(4).to_string())

## Résumé

**Observations clés :**

1. **Distribution** : les ventes suivent une distribution asymétrique (right-skewed), log-transformation utile
2. **Saisonnalité** : pattern hebdomadaire fort (lundi vs samedi), saisonnalité mensuelle (décembre)
3. **Promotions** : augmentent les ventes de ~30-40%, effet variable selon le type de magasin
4. **Météo** : corrélation faible mais non nulle, la pluie réduit légèrement les ventes
5. **Jours fériés** : les ventes augmentent dans les jours précédant un férié
6. **Types de magasins** : forte hétérogénéité, les modèles doivent capturer les profils individuels
7. **Clients** : très forte corrélation avec les ventes (mais pas disponible en prédiction)

**Implications pour la modélisation :**
- Les lags et rolling means seront des features dominantes
- Le `day_of_week` est critique
- Les features par magasin (profil, encodage) ajoutent de l'information
- `customers` ne doit PAS être utilisé comme feature (pas connu à l'avance)